<a href="https://colab.research.google.com/github/hnguye6/real-estate-market-classifier/blob/main/notebooks/02_analysis_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

**Problem**

Real estate investment firms need a fast and data-driven way to identify which U.S. metropolitan statistical area (msa) currently favor buyers versus sellers, in order to prioritize where to look for deals. Manually tracking dozens of individual market indicators across hundreds of metros does not scale, and existing tools like Zillow's Market Heat Index report a single score per metro without translating it into an actionable buy/hold/avoid signal.

**Task**

My task was to build a classification pipeline (from raw Zillow data through SQL-based integration to a validated machine learning model) that assigns each of roughly 900 U.S. metros a monthly Buyer's, Balanced, or Seller's Market label, while explicitly correcting for the housing market's broader boom-bust cycle so classifications reflect a metro's relative character rather than simply which year it is.

# Load the clean table

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

In [ ]:
df = pd.read_csv("clean_real_estate.csv")
df.head(5)

In [ ]:
# Change the date variable from object to datetime
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
# Check for null values
df.info()

In [ ]:
# Observe the distribution of each variable
df.describe()

# Set up SQL

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')

df.to_sql('df',conn,if_exists='replace',index=False)

In [ ]:
def run_sql(sql):
  """Run a SQL query and return results as a pandas DataFrame."""
  return pd.read_sql(sql,conn)

In [ ]:
# Test SQL in Google Colab
run_sql("""
    SELECT *
    FROM df
    LIMIT 3
""")

,RegionID,SizeRank,RegionName,RegionType,StateName,Date,MarketHeatIndex,ZHVI,SaleToListRatio,Year
0,394913,1,"New York, NY",msa,NY,2018-03-31 00:00:00,55.0,478240.041060,0.975858,2018
1,753899,2,"Los Angeles, CA",msa,CA,2018-03-31 00:00:00,67.0,626663.234077,0.998746,2018
2,394463,3,"Chicago, IL",msa,IL,2018-03-31 00:00:00,51.0,235230.901439,0.968852,2018


# Variables' definitions

1. **RegionID:** Zillow's unique numeric identifier for a geographic region (metro or the U.S. national aggregate).
2. **SizeRank:** An ordinal rank of the region by population/urbanization size — lower numbers = larger and more populous metros.
3. **RegionName:** Human-readable name of the region, e.g. "New York, NY".
4. **RegionType:** The geography level of the row (e.g. "msa" or metropolitan statistical area).
5. **StateName:** Two-letter state abbreviation associated with the metro (e.g. "NY", "CA", etc).
6. **Date:** The month-end date the observation applies to (Zillow publishes most series as end-of-month snapshots).
7. **MarketHeatIndex:**  A time series dataset that aims to capture the balance of for-sale supply and demand in a given market. A higher number means the market is more tilted in favor of sellers. It relies on a combination of engagement and listing performance inputs to provide insights into current market dynamics. It is calculated for single-family and condo homes.
8. **ZHVI (Zillow Home Value Index):** A measure of the typical home value and market changes across a given region and housing type. It reflects the typical value for homes in the 35th to 65th percentile range. Available as a smoothed, seasonally adjusted measure and as a raw measure.
9. **Sale-to-List Ratio (mean/median):** Ratio of sale vs. final list price.
10. **Year:** The year extracted from Date column.





**Note on feature scope:** Zillow also publishes *Share of Listings with a Price Cut* and *Mean Days to Pending* at the metro level. Both were excluded from this analysis during an earlier data-cleaning phase (see `real_estate_merge_clean.ipynb`): Price Cut Share is one of the three direct inputs to Zillow's Market Heat Index formula, making it a data-leakage risk rather than an independent predictor, while Mean Days to Pending was both conceptually adjacent to another formula input (share of listings pending within 21 days) and had severe, non-random missingness (48%) concentrated in smaller, lower-liquidity metros. The three variables above — `MarketHeatIndex`, `ZHVI`, and `SaleToListRatio` — are the complete, leakage-free feature/target set used throughout this notebook.

# EDA

## Date

In [ ]:
# Look at the earliest and latest date in this dataset
print(f'The earliest date: {min(df['Date'])}')
print(f'The latest date: {max(df['Date'])}')

## MarketHeatIndex

In [ ]:
# Look at the overall distribution of MarketHeatIndex through histogram
plt.figure(figsize=(10,6))
plt.hist(df['MarketHeatIndex'],bins=30,color='steelblue',edgecolor='white')
plt.title("Distribution of Market Heat Index (2018 - 2026)")
plt.xlabel("Market Heat Index")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Look at the summary statistics of MarketHeatIndex
df['MarketHeatIndex'].describe()

In [ ]:
# Observe the MarketHeatIndex < 0 and MarketHeatIndex > 100
index_outlier = run_sql("""
  SELECT
    'Below 0' AS Outlier_type,
    COUNT(*) AS Count
  FROM df
  WHERE MarketHeatIndex < 0
  UNION ALL
  SELECT
    'Above 100' AS Outlier_type,
    COUNT(*) AS Count
  FROM df
  WHERE MarketHeatIndex > 100
""")
index_outlier


In [ ]:
# Generate a table to show records that contain MarketHeatIndex's outliers
df[(df['MarketHeatIndex'] < 0) | (df['MarketHeatIndex'] > 100)][['RegionName','SizeRank','Date','MarketHeatIndex']].sort_values('MarketHeatIndex')

In [ ]:
# Segment the number of MarketHeatIndex's outliers by Size Rank
df['OutOfRange'] = (df['MarketHeatIndex'] < 0) | (df['MarketHeatIndex'] > 100)
df.groupby(pd.qcut(df['SizeRank'], 4), observed=True)['OutOfRange'].mean()

In [ ]:
# Average Market Heat Index by Year
IndexByYear = df.groupby('Year')['MarketHeatIndex'].mean()
plt.figure(figsize=(10,6))
plt.plot(IndexByYear.index, IndexByYear.values, marker='o')
plt.xlabel('Year')
plt.ylabel('Market Heat Index')
plt.title('Average Market Heat Index by Year')
plt.show()

In [ ]:
# Observe the number of out of range by year
df.groupby('Year')['OutOfRange'].mean()

In [ ]:
# Check the top 20 average MarketHeatIndex per metro
top_metro = run_sql("""
    SELECT
      RegionName AS 'Region Name',
      ROUND(AVG(MarketHeatIndex),2) AS 'Avg. Market Heat Index'
    FROM df
    GROUP BY RegionName
    ORDER BY AVG(MarketHeatIndex) DESC
    LIMIT 20
""")

plt.figure(figsize=(10,6))
sns.barplot(x='Region Name',y='Avg. Market Heat Index',data=top_metro,hue='Region Name', legend=False)
plt.title("Top 20 Regions by Average Market Heat Index")
plt.xlabel('Region Name')
plt.ylabel('Avg. Market Heat Index')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Check the spread of average MarketHeatIndex per metro
top_metro = run_sql("""
    SELECT
      RegionName AS 'Region Name',
      ROUND(AVG(MarketHeatIndex),2) AS 'Avg. Market Heat Index'
    FROM df
    GROUP BY RegionName
    ORDER BY AVG(MarketHeatIndex) DESC
""")
plt.figure(figsize=(10,6))
plt.hist(top_metro['Avg. Market Heat Index'],bins=30,color='orange',edgecolor='white')
plt.title("Distribution of Average Market Heat Index by metro area (2018 - 2026)")
plt.xlabel("Avg. Market Heat Index")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

**Key EDA Takeaways:**

- **Distribution:** `MarketHeatIndex` is right-skewed with a median of 56 (Q1=45, Q3=70), broadly consistent with Zillow's own published tiers (44-55 "neutral", 55-69 "seller's market").
- **Out-of-range values are a large-metro, pandemic-era phenomenon, not a small-metro noise artifact.** About 2.9% of records (1,246 rows) fall outside Zillow's stated "typical" 0-100 range. Contrary to the initial assumption that this would concentrate in small and thinly-traded metros, out-of-range rates are actually *highest* in the largest metro quartile (4.8%) and spike dramatically in 2020-2022 (peaking at 9.0% in 2021) — evidence of genuine, pandemic-driven volatility in large and high-demand metros rather than a data quality issue.
- **A strong national boom-bust cycle dominates the data.** Average `MarketHeatIndex` rose from ~45 in 2018 to a peak of ~76 in 2021, then declined back to ~48 by 2025. Left uncorrected, this cycle would confound any fixed-threshold classification (nearly every metro would look like a "seller's market" in 2021, regardless of its own character). This motivated using within-month and relative tercile-binning for the target label instead of a fixed historical threshold.
- **Top metros mix expected and unexpected names.** The 20 metros with the highest average heat index include well-known high-cost tech hubs (San Francisco, San Jose, Seattle, Boston) alongside several smaller Northeast/Midwest metros (Rochester NY, Chillicothe OH, Buffalo NY), suggesting localized tightness in smaller markets can rival major metros on this metric.
- **Genuine cross-sectional variation exists.** Average `MarketHeatIndex` per metro spans roughly 0 to 125 across the dataset, confirming there's real signal to classify — not a single, uniform national condition.

## Tercile split for Market Heat Index

**What does tercile split actually does?**

Imagine you line up all ~900 metros for a single month, sorted from lowest MarketHeatIndex to highest. A tercile split just cuts that lineup into three equal-sized groups: the bottom third (relatively coolest metros that month → "Buyer's Market"), the middle third ("Balanced"), and the top third (relatively hottest → "Seller's Market"). "Tercile" just means "one of three equal parts" — same idea as a median (which splits into 2 halves), but into 3 groups instead of 2.

**Why we do this separately per month, not once across all 8 years?**

If you sorted and split all rows from 2018-2026 together, 2021's nationwide housing frenzy would dominate — nearly every metro from 2021 would land in "Seller's Market" almost regardless of that metro's own character, since 2021 was hot everywhere.

By re-doing the split fresh for each individual month, you're always comparing a metro to its true peers at that same point in time — so a metro can only be labeled a relative buyer's market if it's cooler than most other metros that month, correcting for the pandemic cycle you found in your EDA.


In [ ]:
def tercile_split(group):
    return pd.qcut(
        group,
        q=3,
        labels=['Buyer\'s Market', 'Balanced Market', 'Seller\'s Market']
    )

df['MarketClass'] = df.groupby('Date')['MarketHeatIndex'].transform(tercile_split)

# Sanity check: classes should come out roughly balanced (~33% each) by construction
df['MarketClass'].value_counts(normalize=True)

# Model Bulding

**Methods used in this section (at a glance):**

- **Multinomial logistic regression** to classify each metro-month into Buyer's / Balanced / Seller's Market, using a **time-based train/test split** (not random shuffling) to avoid leaking information across highly-correlated adjacent months for the same metro.
- **Feature standardization** (`StandardScaler`, fit on train only) so coefficients are comparable across features on very different scales (e.g. home prices vs. ratios).
- **Evaluation via classification report and confusion matrix** (precision/recall/F1 per class), since accuracy alone can hide poor performance on individual classes.
- **Variance Inflation Factor (VIF)** to confirm the features are not redundant with each other before trusting individual coefficients.
- **Feature engineering (within-month percentile rank)** to fix a mismatch between the relative (within-month) target and the originally absolute-valued features — this measurably improved model performance and fixed a systematic prediction bias.
- **Odds ratio interpretation** of the final model's coefficients, to explain feature impact in plain, business-relevant terms.
- **K-means clustering + Adjusted Rand Index** as an independent, unsupervised check on whether the tercile-based classes reflect real structure in the data.
- **Held-out validation** on the single most recent, genuinely out-of-sample month, to test real-world generalization rather than relying on the aggregate test set alone.

## Feature/target setup

In [ ]:
# Step 1: Define features and target deliberately
feature_cols = ['ZHVI', 'SaleToListRatio', 'SizeRank']
X = df[feature_cols]
y = df['MarketClass']  # the 3-class label

# Step 2: Time-based split — train on earlier data and test on the most recent 12 months (May 1, 2025 to April 30, 2026)
cutoff_date = df['Date'].max() - pd.DateOffset(months=12) # Cutoff date is April 30, 2025
train_mask = df['Date'] <= cutoff_date # Dates before and include April 30, 2025
test_mask = df['Date'] > cutoff_date # Dates after April 30, 2025

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# Step 3: Scale features — fit ONLY on train to avoid leaking test-set info into the scaler
scaler = StandardScaler() # StandardScaler transforms your numeric variables so they all have the same scale.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 4: Train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Step 5: Evaluate
y_pred = model.predict(X_test_scaled)

In [ ]:
print("Test set class balance:")
print("")
print(y_test.value_counts(normalize=True))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=model.classes_))
print("Labels order:", model.classes_)

**Understanding precision, recall, and f1-score**
1. **Precision** explains that of all the predicted positives, how many were actually positive. For example, for Balanced Market, of all predicted Balanced Market, how many were actually Balanced Market. In this case, only 52% (or 0.52) were actually Balanced Market
2. **Recall** explains that of all actual positives, how many did we catch. For example, for Balanced Market, of all the actual Balanced Market, how many did our model catch. In this case, it was 39% (0.39)
3. **F-1 score** is single metric that balances Precision and Recall, useful when you need one number instead of two (especially on imbalanced data where Accuracy is misleading.

**Model v1 — Key Takeaways**

- Test accuracy: 60% (versus a 33% random-guess baseline for three classes) — a real signal, but far from strong.
- The model shows a clear, systematic bias: Buyer's Market recall is very high (0.90) but precision is low (0.56) — meaning the model over-predicts Buyer's Market, mislabeling many true Balanced/Seller's Market rows in the process.
- Seller's Market shows the opposite pattern (precision 0.83, recall only 0.47) — the model is conservative about calling something a seller's market and misses over half of the true cases.
- Balanced Market performs worst on both counts (precision 0.52, recall 0.39), consistent with it being the hardest class to separate — the middle, transition zone bordering the other two.
- This systematic imbalance, not just the accuracy number, is the real diagnostic signal here. It motivates the two checks that follow: confirming the features aren't redundant with each other (VIF, below), and investigating whether the features are internally consistent with how the target was constructed (percentile-rank engineering, further below).

## VIF

**Variance Inflation Factor (VIF)** measures how much a feature's variance is "inflated" because it is correlated with other features, rather than being independent information.

A VIF of 1 means no correlation with other predictors; VIF above 5 (some use 10) signals a multicollinearity problem.

In [ ]:
# Use the unscaled training features — VIF is unaffected by simple linear scaling
X_vif = add_constant(X_train)

vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data)

The three features that actually matter — ZHVI (1.12), SaleToListRatio (1.24), SizeRank (1.12) — are all very close to 1, meaning no multicollinearity between them. This matches earlier correlation matrix suggested (weak correlations among these three), and it is good news: the logistic regression coefficients should be individually stable and safe to interpret

## Percentile rank within month

**What is it?**

Our target (MarketClass) was built by ranking each metro against its peers within the same month, so a metro is only labeled "hot" or "cold" relative to that specific point in the housing cycle.

**Why are we doing it?**

Our original features (ZHVI, SaleToListRatio) were left as absolute values, meaning they carried no sense of "relative to this month" — this mismatch was the likely cause of our first model's systematic bias toward over-predicting Buyer's Market.

To fix this, we converted ZHVI and SaleToListRatio into within-month percentile ranks (a value from 0 to 1, where 1 means the highest value among all metros that month), computed using groupby('Date').rank(pct=True). This puts our features on the same "relative to peers at that time" basis as our target, so the model can learn genuine cross-sectional differences between metros rather than trying to infer relative rank from raw, un-relativized numbers.

In [ ]:
df['ZHVI_pctrank'] = df.groupby('Date')['ZHVI'].rank(pct=True)
df['SaleToListRatio_pctrank'] = df.groupby('Date')['SaleToListRatio'].rank(pct=True)

df.head(5)

**Retrain the model based on percentile rank within month features**

In [ ]:
feature_cols = ['ZHVI_pctrank', 'SaleToListRatio_pctrank', 'SizeRank']
X = df[feature_cols]
y = df['MarketClass']

# same time-based split as before
cutoff_date = df['Date'].max() - pd.DateOffset(months=12)
train_mask = df['Date'] <= cutoff_date
test_mask = df['Date'] > cutoff_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_v2 = LogisticRegression(max_iter=1000, random_state=42)
model_v2.fit(X_train_scaled, y_train)

y_pred_v2 = model_v2.predict(X_test_scaled)

print(classification_report(y_test, y_pred_v2))
print(confusion_matrix(y_test, y_pred_v2, labels=model_v2.classes_))
print("Labels order:", model_v2.classes_)

**Model v2 — Key Takeaways (Before vs. After Feature Engineering)**

| Metric | Model v1 (absolute features) | Model v2 (percentile-rank features) |
|---|---|---|
| Accuracy | 60% | 62% |
| Seller's Market recall | 47% | 75% |
| Buyer's Market precision | 56% | 64% |
| Macro avg F1 | 0.58 | 0.61 |

- Re-aligning the features to the same "relative to this month" basis as the target directly fixed the bias diagnosed in Model v1: Seller's Market recall jumped from 47% to 75% (correct Seller's predictions rose from 990 to 1,588), and Buyer's Market precision improved from 56% to 64% (fewer Balanced-Market rows misclassified as Buyer's).
- **Balanced Market remains the weakest class** (F1 essentially unchanged, 0.45→0.46), but this is expected rather than concerning — it's the middle, transition zone bordering both other classes, and its prediction errors are now split more evenly between Buyer's and Seller's rather than leaking mostly in one direction.
- This before/after comparison is solid evidence that the feature-target mismatch diagnosed after Model v1 was the right hypothesis, and that fixing it meaningfully improved the model — a complete diagnose → hypothesize → fix → validate cycle.

In [ ]:
print(model_v2.classes_)
print(model_v2.coef_)
print(model_v2.coef_.shape)

In [ ]:
coef_df = pd.DataFrame(
    model_v2.coef_,
    index=model_v2.classes_,
    columns=feature_cols
)

odds_ratios = np.exp(coef_df)
print(odds_ratios)

SaleToListRatio_pctrank is by far the dominant predictor: a 1-standard-deviation increase in a metro's within-month sale-to-list ratio percentile multiplies the odds of a Seller's Market classification by roughly 3.63x, and cuts the odds of a Buyer's Market classification to about 0.28x (a ~72% reduction), holding other features constant. This aligns with real-world intuition — homes selling at or above asking price is a textbook signal of seller-favorable conditions.

ZHVI_pctrank and SizeRank had comparatively weak, near-neutral effects (odds ratios between 0.92 and 1.05 across all classes), suggesting home value tier and metro size contribute little independent signal once sale-to-list behavior is accounted for. Note that all odds ratios are per 1 standard deviation of the (scaled) feature, not per raw unit.

**Held-out recent-month validation**

In [ ]:
# Step 1: latest date
latest_date = df['Date'].max()

In [ ]:
# Step 2: boolean mask (aligned by position to X_test's rows) for that month
latest_mask = (df.loc[X_test.index, 'Date'] == latest_date).values

# Step 3: build the comparison table
comparison = df.loc[X_test.index[latest_mask], ['RegionName', 'MarketHeatIndex']].copy()
comparison['TrueClass'] = y_test.values[latest_mask]
comparison['PredictedClass'] = y_pred_v2[latest_mask]

comparison = comparison.sort_values('MarketHeatIndex', ascending=False)

print(f"Accuracy for {latest_date.date()} only: {(comparison['TrueClass'] == comparison['PredictedClass']).mean():.3f}")
print(f"Number of metros in this month: {len(comparison)}")
comparison.head(20)

In [ ]:
comparison2 = df.loc[X_test.index[latest_mask], ['RegionName', 'SizeRank', 'MarketHeatIndex']].copy()
comparison2['TrueClass'] = y_test.values[latest_mask]
comparison2['PredictedClass'] = y_pred_v2[latest_mask]
comparison2['Correct'] = comparison2['TrueClass'] == comparison2['PredictedClass']

size_rank_chart = comparison2.groupby(pd.qcut(comparison2['SizeRank'], 4), observed=True)['Correct'].mean()

In [ ]:
size_rank_chart

In [ ]:
size_rank_chart.index = ['Large Metro', 'Mid-Large Metro', 'Mid-Small Metro', 'Small Metro']

In [ ]:
import matplotlib.ticker as mtick

plt.figure(figsize=(10, 6), facecolor='#F2F1EF')
ax = sns.barplot(x=size_rank_chart.index, y=size_rank_chart.values,
                 color='#6b6361', legend=False)

# Add percentage labels on top of each bar
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.2%}',
            ha='center', va='bottom', fontsize=10 * 1.2)  # 👈 20% larger

ax.set_facecolor('#F2F1EF')
plt.title("Model Accuracy by City Size Rank", fontsize=12 * 1.5)   # 👈 50% larger
plt.xlabel("City Size Rank", fontsize=10 * 1.2)                     # 👈 20% larger
plt.ylabel("Accuracy", fontsize=10 * 1.2)                           # 👈 20% larger
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right', fontsize=10 * 1.2)              # 👈 20% larger
plt.yticks(fontsize=10 * 1.2)                                        # 👈 20% larger
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=2))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6), facecolor='#F2F1EF')
ax = sns.barplot(x=size_rank_chart.index, y=size_rank_chart.values,
                 color='#6b6361', legend=False)

# Add percentage labels on top of each bar
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.2%}',
            ha='center', va='bottom', fontsize=10 * 1.2)  # 👈 20% larger

ax.set_facecolor('#F2F1EF')
plt.xlabel("City Size Rank", fontsize=10 * 1.2)                     # 👈 20% larger
plt.ylabel("Accuracy", fontsize=10 * 1.2)                           # 👈 20% larger
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right', fontsize=10 * 1.2)              # 👈 20% larger
plt.yticks(fontsize=10 * 1.2)                                        # 👈 20% larger
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=2))
plt.tight_layout()
plt.show()

Overall accuracy for the most recent, fully out-of-sample month was 58.9%, close to the 62% test-set average.

Breaking accuracy down by metro size reveals a strong, monotonic pattern: 79.9% for the largest quartile of metros, declining to 44.5% for the smallest quartile. This mirrors data-quality patterns found earlier in the project (missing SaleToListRatio values and MarketHeatIndex volatility were both concentrated in smaller metros), reinforcing that the model's reliability is inherently tied to market size and liquidity — a practical caveat for how confidently the tool's output should be used for smaller markets.

## K-means

Unlike logistic regression, K-means does not use MarketClass labels at all. Instead, it looks at each metro-month's position in feature space (based on ZHVI_pctrank, SaleToListRatio_pctrank, SizeRank) and groups the data into 3 clusters based purely on which points sit close together.

It has no idea which cluster "should" be Buyer's/Balanced/Seller's — it just finds natural groupings. Comparing those groupings to the actual labels tells you whether the classes you defined via tercile-splitting also show up as naturally separable clusters in the data, or whether they only exist because you imposed them.

In [ ]:
# Scale the full feature set — K-means is distance-based, so features need to be on comparable scales
X_full = df[feature_cols]
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_full)

# Fit K-means with 3 clusters, since we're comparing to your 3-class target
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['KMeansCluster'] = kmeans.fit_predict(X_full_scaled)

# Compare: does each K-means cluster line up with a particular MarketClass?
comparison = pd.crosstab(df['MarketClass'], df['KMeansCluster'])
print(comparison)

**Output's interpretation:**
1. Cluster 0 (15,528 total): mostly Buyer's Market (8,930 = 57.5%), with meaningful Balanced Market overlap (4,279 = 27.6%).
2. Cluster 1 (19,457 total): mostly Seller's Market (10,037 = 51.6%), but also a lot of Balanced Market bleeding in (7,248 = 37.3%).
3. Cluster 2 (7,859 total, notably smaller): a mix, leaning Buyer's/Balanced again (3,767 and 2,732).

Two things stand out:
1. First, K-means roughly recovers a Buyer's-leaning group and a Seller's-leaning group — so there is real, natural structure in the data that echoes my labels, which is reassuring.
2. Second, and more interesting: Balanced Market never dominates any single cluster — its 14,259 rows are smeared fairly evenly across all three clusters (4,279 / 7,248 / 2,732) rather than forming its own distinct group.

This actually converges nicely with what my logistic regression already told me: Balanced Market was the weakest and hardest-to-separate class there too. It makes intuitive sense: "balanced" is the transition zone between buyer-favorable and seller-favorable conditions, not a distinct market character of its own — so it is reasonable that it does not form a separate natural cluster, rather than a flaw in your models.

**Adjusted Rand Index**

To make this rigorous rather than eyeballed, compute the Adjusted Rand Index, which measures how well the clusters align with true labels (0 = no better than random grouping, 1 = perfect match), and correctly handles the fact that cluster numbers are arbitrary:

In [ ]:
ari = adjusted_rand_score(df['MarketClass'],df['KMeansCluster'])
print(f'Adjusted Rand Index: {ari:.3f}')

The resulting Adjusted Rand Index of 0.137 indicates weak alignment, driven largely by two factors: our tercile split forces exactly equal-sized classes (33% each) while K-means naturally found unequal-sized groups (36%, 45%, and 18% of the data), and the Balanced Market class in particular did not correspond to any single dominant cluster, instead spreading fairly evenly across all three.

This suggests that "balanced" functions more as a transition zone between buyer- and seller-favorable conditions than as a distinct market character — a finding that reinforces the logistic regression results, where Balanced Market was similarly the hardest class to predict.

# Data Manipulation (for Tableau)

In [ ]:
# Step 1: Pull the columns you need for the map, reusing the latest-month mask from Day 5
export_df = df.loc[X_test.index[latest_mask], ['RegionName', 'StateName', 'SizeRank', 'MarketHeatIndex']].copy()
export_df['TrueClass'] = y_test.values[latest_mask]
export_df['PredictedClass'] = y_pred_v2[latest_mask]

# Step 2: Split "City, ST" into just the city name — Tableau's City geographic role
# expects a plain city name, paired with the State field to disambiguate (e.g. Springfield, IL vs Springfield, MO)
export_df['City'] = export_df['RegionName'].str.split(',').str[0]

In [ ]:
export_df.head(5)

In [ ]:
# Step 3: Save to CSV
export_df.to_csv('tableau_market_map.csv', index=False)